# local-db-test: local database demo with "SQLite"

for better (Rust-based) implementtion (10x+ speed increase) look at:

- [duckdb](https://duckdb.org/)
- [Polars vs Pandas](https://pola.rs/)

In [1]:
import logging
import sys
import subprocess
from pathlib import Path

In [2]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

In [3]:
from dve.config import conf

In [4]:
logging.basicConfig(stream=sys.stderr, level=logging.DEBUG)

log = logging.getLogger(__name__)

In [5]:
cnf = conf.get_config()

In [6]:
local_uri = conf.get_config().db("demo").uri()

In [7]:
print(f"local_uri: {conf.get_config().db('demo').dump()}")
print(f"local_dsc: {conf.get_config().db('demo').dump(full=True)}")

local_uri: sqlite+pysqlite:///data/ext/test/demo/demo.db
local_dsc: {
    "alias": "demo_lt"
}


In [8]:
! pwd

/root/work/vb/dve-sample-r/notebooks/sys-test


In [9]:
# jupyter current dir ("./notebooks") adjustment
local_uri = local_uri.replace("///", "///../../")

In [10]:
print(local_uri)

sqlite+pysqlite:///../../data/ext/test/demo/demo.db


In [11]:
def sql_bind(engine):
    def sql_exec(qry: str):
        # engine.begin() automatically commits the transaction on success
        with engine.begin() as con:
            rs = con.execute(text(qry))

        # safely check if the result set is iterable (e.g., SELECT queries)
        if rs.returns_rows:
            for row in rs:
                print(row)
        else:
            # For CREATE/INSERT/UPDATE, just print the result status
            print(f"Executed successfully. Rows affected: {rs.rowcount}")

    return sql_exec


def drop_database(uri: str):
    # 0. Parse the actual file path from the SQLAlchemy URI automatically
    db_file_path = make_url(uri).database

    if db_file_path is None:
        return None

    db_path = Path(db_file_path)

    # 1. Ensure parent directory exists (creates 'dbms' and any missing parent directories)
    db_path.parent.mkdir(parents=True, exist_ok=True)

    # 2. Delete main database file and temporary WAL/SHM journal files if present
    db_path.unlink(missing_ok=True)
    Path(f"{db_path}-wal").unlink(missing_ok=True)
    Path(f"{db_path}-shm").unlink(missing_ok=True)

    # 3. Ensure the parent directory exists
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return db_path

In [12]:
db_path = drop_database(local_uri)

In [13]:
engine = create_engine(local_uri, echo=True)

In [14]:
sql_exec = sql_bind(engine)

In [15]:
sql_exec("SELECT current_timestamp")

2026-10-02 03:52:58,818 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:58,819 INFO sqlalchemy.engine.Engine SELECT current_timestamp


INFO:sqlalchemy.engine.Engine:SELECT current_timestamp


2026-10-02 03:52:58,819 INFO sqlalchemy.engine.Engine [generated in 0.00057s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00057s] ()


2026-10-02 03:52:58,820 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('2026-10-02 01:52:58',)


In [16]:
schema_sql = """

SELECT
  m.name as table_name,
  p.name as column_name
FROM
  sqlite_master AS m
JOIN
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table'
ORDER BY
  m.name,
  p.cid

"""

In [17]:
sql_exec(schema_sql)

2026-10-02 03:52:58,852 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:58,853 INFO sqlalchemy.engine.Engine 

SELECT
  m.name as table_name,
  p.name as column_name
FROM
  sqlite_master AS m
JOIN
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table'
ORDER BY
  m.name,
  p.cid




INFO:sqlalchemy.engine.Engine:

SELECT
  m.name as table_name,
  p.name as column_name
FROM
  sqlite_master AS m
JOIN
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table'
ORDER BY
  m.name,
  p.cid




2026-10-02 03:52:58,853 INFO sqlalchemy.engine.Engine [generated in 0.00060s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00060s] ()


2026-10-02 03:52:58,854 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


In [18]:
create_sql = """

CREATE TABLE IF NOT EXISTS demo_table (
        column1 text PRIMARY KEY,
        column2 text NOT NULL,
        column3 integer DEFAULT 0
);

"""

insert_sql = """

INSERT INTO 'demo_table' ('column1', 'column2') VALUES
  ('a', 'data1'),
  ('b', 'data2'),
  ('c', 'data1'),
  ('d', 'data3'),
  ('e', 'data3');

"""

update_sql = """

UPDATE 'demo_table' SET
  column3 = column3 + 1
 WHERE
   column2 in ('data2', 'data3');

"""

select_sql = """

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;

"""

group_sql = """

SELECT column2, sum(column3) as tot3 FROM 'demo_table'
GROUP BY column2 HAVING tot3 > 0
ORDER BY tot3 DESC, column2;

"""

In [19]:
sql_exec(create_sql)

2026-10-02 03:52:58,886 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:58,887 INFO sqlalchemy.engine.Engine 

CREATE TABLE IF NOT EXISTS demo_table (
        column1 text PRIMARY KEY,
        column2 text NOT NULL,
        column3 integer DEFAULT 0
);




INFO:sqlalchemy.engine.Engine:

CREATE TABLE IF NOT EXISTS demo_table (
        column1 text PRIMARY KEY,
        column2 text NOT NULL,
        column3 integer DEFAULT 0
);




2026-10-02 03:52:58,888 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00102s] ()


2026-10-02 03:52:58,924 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Executed successfully. Rows affected: -1


In [20]:
sql_exec(schema_sql)

2026-10-02 03:52:58,928 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:58,928 INFO sqlalchemy.engine.Engine 

SELECT
  m.name as table_name,
  p.name as column_name
FROM
  sqlite_master AS m
JOIN
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table'
ORDER BY
  m.name,
  p.cid




INFO:sqlalchemy.engine.Engine:

SELECT
  m.name as table_name,
  p.name as column_name
FROM
  sqlite_master AS m
JOIN
  pragma_table_info(m.name) AS p
WHERE
  m.type = 'table'
ORDER BY
  m.name,
  p.cid




2026-10-02 03:52:58,929 INFO sqlalchemy.engine.Engine [cached since 0.07635s ago] ()


INFO:sqlalchemy.engine.Engine:[cached since 0.07635s ago] ()


2026-10-02 03:52:58,930 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('demo_table', 'column1')
('demo_table', 'column2')
('demo_table', 'column3')


In [21]:
sql_exec(insert_sql)

2026-10-02 03:52:58,950 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:58,951 INFO sqlalchemy.engine.Engine 

INSERT INTO 'demo_table' ('column1', 'column2') VALUES
  ('a', 'data1'),
  ('b', 'data2'),
  ('c', 'data1'),
  ('d', 'data3'),
  ('e', 'data3');




INFO:sqlalchemy.engine.Engine:

INSERT INTO 'demo_table' ('column1', 'column2') VALUES
  ('a', 'data1'),
  ('b', 'data2'),
  ('c', 'data1'),
  ('d', 'data3'),
  ('e', 'data3');




2026-10-02 03:52:58,952 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00082s] ()


2026-10-02 03:52:58,953 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Executed successfully. Rows affected: 5


In [22]:
sql_exec(select_sql)

2026-10-02 03:52:58,979 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:58,980 INFO sqlalchemy.engine.Engine 

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




INFO:sqlalchemy.engine.Engine:

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




2026-10-02 03:52:58,981 INFO sqlalchemy.engine.Engine [generated in 0.00065s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00065s] ()


2026-10-02 03:52:58,982 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('c', 'data1', 0)
('a', 'data1', 0)
('b', 'data2', 0)
('e', 'data3', 0)
('d', 'data3', 0)


In [23]:
sql_exec(update_sql)
sql_exec(update_sql)
sql_exec(update_sql)

2026-10-02 03:52:59,002 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:59,005 INFO sqlalchemy.engine.Engine 

UPDATE 'demo_table' SET
  column3 = column3 + 1
 WHERE
   column2 in ('data2', 'data3');




INFO:sqlalchemy.engine.Engine:

UPDATE 'demo_table' SET
  column3 = column3 + 1
 WHERE
   column2 in ('data2', 'data3');




2026-10-02 03:52:59,006 INFO sqlalchemy.engine.Engine [generated in 0.00133s] ()


...

2026-10-02 03:52:59,070 INFO sqlalchemy.engine.Engine [cached since 0.06527s ago] ()


INFO:sqlalchemy.engine.Engine:[cached since 0.06527s ago] ()


2026-10-02 03:52:59,071 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Executed successfully. Rows affected: 3


In [24]:
sql_exec(select_sql)

2026-10-02 03:52:59,093 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:59,094 INFO sqlalchemy.engine.Engine 

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




INFO:sqlalchemy.engine.Engine:

SELECT * FROM 'demo_table'
ORDER BY column2, column1 DESC;




2026-10-02 03:52:59,094 INFO sqlalchemy.engine.Engine [cached since 0.1141s ago] ()


INFO:sqlalchemy.engine.Engine:[cached since 0.1141s ago] ()


2026-10-02 03:52:59,095 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('c', 'data1', 0)
('a', 'data1', 0)
('b', 'data2', 3)
('e', 'data3', 3)
('d', 'data3', 3)


In [25]:
sql_exec(group_sql)

2026-10-02 03:52:59,123 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-10-02 03:52:59,124 INFO sqlalchemy.engine.Engine 

SELECT column2, sum(column3) as tot3 FROM 'demo_table'
GROUp BY column2 HAVING tot3 > 0
ORDER BY tot3 DESC, column2;




INFO:sqlalchemy.engine.Engine:

SELECT column2, sum(column3) as tot3 FROM 'demo_table'
GROUp BY column2 HAVING tot3 > 0
ORDER BY tot3 DESC, column2;




2026-10-02 03:52:59,125 INFO sqlalchemy.engine.Engine [generated in 0.00066s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00066s] ()


2026-10-02 03:52:59,126 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


('data3', 6)
('data2', 3)


In [26]:
assert db_path
subprocess.run(["ls", "-l", db_path.parent], check=True)

total 12
-rw-r--r-- 1 root root 12288 Oct  2 03:52 demo.db


CompletedProcess(args=['ls', '-l', PosixPath('../../data/ext/test/demo')], returncode=0)